Welcome to automatic segmentation! First we will start by ensuring we have the right packages installed and are using the colabs GPUs.

In [ ]:
import torch

print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU found - using CPU (will be slow)")

# Install required packages
print("\nInstalling packages...")
!pip install -q nnunetv2 SimpleITK nibabel numpy-stl scikit-image pydicom monai rt_utils TotalSegmentator

print("✓ Setup complete!")

This cell ensures that the functions we will be using are imported into this instance, as well as makes the folders we will be using for our segmentation today.

In [ ]:
## import
import os
import numpy as np
import torch
import pydicom
import matplotlib.pyplot as plt
import SimpleITK as sitk
from monai.bundle import ConfigParser, download
from monai.transforms import LoadImage, LoadImaged, Orientation, Orientationd, EnsureChannelFirst, EnsureChannelFirstd, Compose
from rt_utils import RTStructBuilder
from scipy.ndimage import label, measurements
import json
import glob
print("✓ Function import complete!")

# Create input + output directories
os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)
os.makedirs('/content/input/chest_ct', exist_ok=True)

print("✓ Folder setup complete!")

Download the CT model from the IDC Database:

In [ ]:
##Download the model from IDC database
!pip3 install --upgrade idc-index
!idc download 1.3.6.1.4.1.14519.5.2.1.4334.1501.119531128953610472040332469413 --download-dir /content/input/chest_ct

We will view the metadata from our scan, the key here is to look at the dimension (spatial_shape) of the scan, which describes the slices in each dimension (coronal, sagittal, and axial).

In [ ]:
##import CT scan .dcm files from chest_CT folder
data_dir = "/content/input/chest_ct/nsclc_radiogenomics/AMC-015/1.3.6.1.4.1.14519.5.2.1.4334.1501.119531128953610472040332469413/CT_1.3.6.1.4.1.14519.5.2.1.4334.1501.253298261882254993527951068007"
image_loader = LoadImage(image_only=True)
CT = image_loader(data_dir)

##view metadata from scan
CT.meta

Now, view a slice to ensure download happened correctly and files are in the correct location

In [ ]:
## view slice
CT_coronal_slice = CT[:,256].cpu().numpy()
plt.figure(figsize=(6, 4.5))
plt.pcolormesh(CT_coronal_slice.T, cmap='Greys_r')
plt.colorbar(label='HU')
plt.axis('off')
plt.show()

The image is upside down, so we need to reorient the image and add the correct modifacations for the dimensions that will be used for furhter processing.

In [ ]:
##shape image
CT.shape

## add channel dimension
channel_transform = EnsureChannelFirst()
CT = channel_transform(CT)
CT.shape

##re orient and view again
orientation_transform = Orientation(axcodes=('LPS'))
CT = orientation_transform(CT)

##plot
CT_coronal_slice = CT[0,:,256].cpu().numpy()
plt.figure(figsize=(6, 4.5))
plt.pcolormesh(CT_coronal_slice.T, cmap='Greys_r')
plt.colorbar(label='HU')
plt.axis('off')
plt.show()

To use the segmentation tool we have chosen, we must convert our DICOM file to an NIfTI. After the conversion, we can run Total Segmentor and save the output.

In [ ]:
# Convert your DICOM to NIfTI
reader = sitk.ImageSeriesReader()
reader.SetFileNames(reader.GetGDCMSeriesFileNames("/content/input/chest_ct/nsclc_radiogenomics/AMC-015/1.3.6.1.4.1.14519.5.2.1.4334.1501.119531128953610472040332469413/CT_1.3.6.1.4.1.14519.5.2.1.4334.1501.253298261882254993527951068007"))
ct_img = reader.Execute()
sitk.WriteImage(ct_img, "/content/input/ct_scan.nii.gz")

# Run segmentation (takes 2-5 minutes)
!TotalSegmentator -i /content/input/ct_scan.nii.gz -o /content/output/totalseg --fast


From the total segmentation, pull out the heart files and combine them into one object. Select the scan that contains the mose heart pixels.

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# TotalSegmentator saves each organ as a separate file
seg_dir = "/content/output/totalseg"

# Heart structure filenames in TotalSegmentator
HEART_FILES = {
    'heart': 'Heart'
}

# Find and load heart files
print("Looking for heart structures...")
heart_data = {}
ct = nib.load("/content/input/ct_scan.nii.gz")
ct_array = ct.get_fdata()

for filename, label_name in HEART_FILES.items():
    filepath = os.path.join(seg_dir, f"{filename}.nii.gz")
    if os.path.exists(filepath):
        seg_img = nib.load(filepath)
        seg_array = seg_img.get_fdata()
        voxel_count = np.sum(seg_array > 0)
        if voxel_count > 0:
            heart_data[label_name] = seg_array
            print(f"✓ Found {label_name}: {voxel_count:,} voxels")
        else:
            print(f"  {label_name}: file exists but empty")
    else:
        print(f"  {label_name}: not found")


print(f"\n✓ Found {len(heart_data)} heart structures!")

# Combine all heart structures into one array
combined_heart = np.zeros_like(ct_array)
label_colors = {}
current_label = 1

for name, seg_array in heart_data.items():
    combined_heart[seg_array > 0] = current_label
    label_colors[current_label] = name
    current_label += 1

# Find best slice
slice_scores = [np.sum(combined_heart[:,:,i] > 0) for i in range(combined_heart.shape[2])]
best_slice = np.argmax(slice_scores)

print(f"\nBest slice: {best_slice}")


View the best slice of the CT scan, and overlay with the heart segmentation.

In [ ]:
## One slice of the heart segmentation
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# CT windowing
ct_slice = ct_array[:,:,best_slice]
ct_win = np.clip(ct_slice, -1200, 300)
ct_win = (ct_win - ct_win.min()) / (ct_win.max() - ct_win.min())

heart_slice = combined_heart[:,:,best_slice]

# Plot
axes[0].imshow(ct_win.T, cmap='gray', origin='lower')
axes[0].set_title(f'CT - Slice {best_slice}', fontsize=14)
axes[0].axis('off')

axes[1].imshow(heart_slice.T, cmap='nipy_spectral', origin='lower', vmin=0, vmax=len(heart_data))
axes[1].set_title('Heart Segmentation', fontsize=14)
axes[1].axis('off')

axes[2].imshow(ct_win.T, cmap='gray', origin='lower')
masked = np.ma.masked_where(heart_slice.T == 0, heart_slice.T)
axes[2].imshow(masked, cmap='nipy_spectral', alpha=0.6, origin='lower', vmin=0, vmax=len(heart_data))
axes[2].set_title('Overlay', fontsize=14)
axes[2].axis('off')

# Add legend
legend_text = "Heart Structures:\n" + "\n".join([f"{i}: {name}" for i, name in label_colors.items()])
fig.text(0.02, 0.02, legend_text, fontsize=11,
          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('/content/output/heart_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved to /content/output/heart_visualization.png")

# Show stats for this slice
print(f"\nStructures in slice {best_slice}:")
for label_id, name in label_colors.items():
    pixels = np.sum(heart_slice == label_id)
    if pixels > 0:
        print(f"  {name}: {pixels} pixels")


Check each and every slice that includes the heart segmentation.

In [ ]:
# Create multi-slice view
print("\nCreating multi-slice view...")
slices_with_heart = [i for i in range(combined_heart.shape[2]) if np.sum(combined_heart[:,:,i] > 0) > 100]

if len(slices_with_heart) >= 12:
    indices = np.linspace(0, len(slices_with_heart)-1, 12, dtype=int)
    selected_slices = [slices_with_heart[i] for i in indices]

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()

    for idx, slice_idx in enumerate(selected_slices):
        ct_s = np.clip(ct_array[:,:,slice_idx], -1200, 300)
        ct_s = (ct_s - ct_s.min()) / (ct_s.max() - ct_s.min())

        axes[idx].imshow(ct_s.T, cmap='grey', origin='lower')
        masked = np.ma.masked_where(combined_heart[:,:,slice_idx].T == 0, combined_heart[:,:,slice_idx].T)
        axes[idx].imshow(masked, cmap='nipy_spectral', alpha=0.6, origin='lower')
        axes[idx].set_title(f'Slice {slice_idx}', fontsize=10)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig('/content/output/heart_multislice.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Multi-slice view saved!")

Final Step!! Exporting the 3D model of a heart to an STL for further processing and 3D printing.

In [ ]:
import nibabel as nib
import numpy as np
from skimage.measure import marching_cubes
from stl import mesh

input_path = '/content/output/totalseg/heart.nii.gz'
output_path = '/content/output/heart.stl'

# 1. Load the NIfTI file
nifti_img = nib.load(input_path)
data = nifti_img.get_fdata()

# 2. Extract the Voxel Spacings (The Magic Step)
# This gets the size of voxels in mm (x, y, z) from the header
# Example: (0.35, 0.35, 3.0) -> This tells us the Z slices are thick!
header = nifti_img.header
zooms = header.get_zooms()

print(f"Data Shape: {data.shape}")
print(f"Voxel Spacing (mm): {zooms}")

# 3. Handle Segmentation Labels
# If your segmentation uses '1' for femur, '2' for tibia, pick one.
# If it's a binary mask (0 and 1), you are good to go.
# Let's assume we want to mesh everything that isn't background (0)
mask = data > 0

# 4. Run Marching Cubes WITH spacing
# step_size=1 ensures high quality. Increase to 2 or 3 for smaller file size.
verts, faces, normals, values = marching_cubes(mask, level=0, spacing=zooms)

# 5. Create the mesh object
obj_mesh = mesh.Mesh(np.zeros(faces.shape[0], dtype=mesh.Mesh.dtype))
for i, f in enumerate(faces):
    for j in range(3):
        obj_mesh.vectors[i][j] = verts[f[j],:]

# 6. Save
obj_mesh.save(output_path)
print(f"Saved corrected STL to {output_path}")

Ensure you download the .stl file from content/output/ before closing this instance of colabs, otherwise you will need to repeat the entire process.